# UNSW-NB15 GNN (edge classification)

**Flow:** load -> build one graph per split -> look at the graph -> train an E-GraphSAGE GNN -> evaluate on the held-out test graph -> fuse with the tabular models.

**Idea:** every network flow is an **edge** between two endpoints. The GNN classifies each edge (attack type) using its own 45 features *plus* the features of the flows that touch the same endpoints - a signal the tabular baselines never see.

Rules followed:
- The train graph is built from train flows only, the test graph from test flows only (inductive). Test flows never take part in training.
- No labels are used in message passing - only edge features. Labels appear only in the loss.
- Same features, same test set and same metrics as the baseline notebook, so the numbers are directly comparable.

In [ ]:
import copy
import glob
import time

import joblib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

DEBUG = False   # True = quick smoke test on a small random sample
SEED = 42
DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN = 64
FANOUT_1, FANOUT_2 = 8, 5   # neighbours sampled per node in layer 2 / layer 1
BATCH = 2048
MAX_EPOCHS = 2 if DEBUG else 12
PATIENCE = 3
LR = 2e-3

torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", DEVICE, "| debug:", DEBUG)

_fig_count = 0
_plt_show = plt.show


def show_and_save(*args, **kwargs):
    """Show every figure and also keep it as a PNG in the notebook output."""
    global _fig_count
    _fig_count += 1
    plt.savefig(f"/kaggle/working/fig_{_fig_count:02d}.png", dpi=130, bbox_inches="tight")
    _plt_show(*args, **kwargs)


plt.show = show_and_save

## 1. Load data

`train.parquet` / `test.parquet` hold the edge features and `label` (original class balance - SMOTE rows are synthetic and have no real endpoints, so they can't be graph edges). `train_graph.csv` / `test_graph.csv` hold the endpoints (IPs, ports), joined by `row_id`.

In [ ]:
train_df = pd.read_parquet(f"{DATASET_PATH}/train.parquet")
test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet")
if DEBUG:
    train_df, test_df = train_df.sample(150_000, random_state=SEED), test_df.sample(30_000, random_state=SEED)

train_graph = pd.read_csv(f"{DATASET_PATH}/train_graph.csv", index_col="row_id").loc[train_df.index]
test_graph = pd.read_csv(f"{DATASET_PATH}/test_graph.csv", index_col="row_id").loc[test_df.index]

class_names = joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_
n_classes = len(class_names)
normal_label = list(class_names).index("Normal")
feature_cols = [c for c in train_df.columns if c != "label"]
print("train edges", len(train_df), "| test edges", len(test_df), "| edge features", len(feature_cols))

## 2. Build the graphs

**Nodes = (IP, port) endpoints.** The dataset has only 49 distinct IPs, so an IP-only graph would be 49 nodes with millions of edges each - averaging over that tells the model nothing. IP:port gives roughly a million nodes: servers become hubs, short-lived client ports stay small.

Each graph is stored as a CSR neighbour list (for every node: its incident edges and the node at the other end) so neighbours can be sampled quickly.

In [ ]:
def build_graph(graph_df, feat_df):
    n = len(graph_df)
    ip_code, ip_names = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
    ip_code = ip_code.astype("int64")
    ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
    node_id, node_keys = pd.factorize(ip_code * 70_000 + ports)   # one node per distinct (IP, port)
    src, dst = torch.from_numpy(node_id[:n]), torch.from_numpy(node_id[n:])
    n_nodes = len(node_keys)

    nodes = torch.cat([src, dst])          # every edge appears once per endpoint
    other = torch.cat([dst, src])
    edge_ids = torch.cat([torch.arange(n), torch.arange(n)])
    order = torch.argsort(nodes)
    ptr = torch.zeros(n_nodes + 1, dtype=torch.long)
    ptr[1:] = torch.cumsum(torch.bincount(nodes, minlength=n_nodes), 0)
    return {
        "n_nodes": n_nodes, "src": src, "dst": dst,
        "ptr": ptr, "inc_edge": edge_ids[order], "inc_other": other[order],
        "node_name": lambda v: f"{ip_names[node_keys[v] // 70_000]}:{node_keys[v] % 70_000 - 1}",
        "x": torch.from_numpy(feat_df[feature_cols].to_numpy("float32")),
        "y": torch.from_numpy(feat_df["label"].to_numpy("int64")),
    }


def sample_neighbors(G, nodes, fanout):
    """Draw `fanout` incident edges (with replacement) for each node -> (edge ids, neighbour node ids)."""
    start = G["ptr"][nodes]
    deg = G["ptr"][nodes + 1] - start
    offset = (torch.rand(len(nodes), fanout, device=nodes.device) * deg.unsqueeze(1)).long()
    offset = torch.minimum(offset, deg.unsqueeze(1) - 1)
    pos = start.unsqueeze(1) + offset
    return G["inc_edge"][pos], G["inc_other"][pos]


G_train, G_test = build_graph(train_graph, train_df), build_graph(test_graph, test_df)
for name, G in [("train", G_train), ("test", G_test)]:
    deg = (G["ptr"][1:] - G["ptr"][:-1]).numpy()
    print(f"{name}: {len(G['src'])} edges, {G['n_nodes']} nodes | degree median {int(np.median(deg))}, "
          f"max {deg.max()}, nodes with degree 1: {(deg == 1).mean():.1%}")

## 3. What the graph looks like

Five views of the **train** graph. Edge colour = attack class of the flow. Every figure is also saved as a PNG in the notebook output.

In [ ]:
palette = dict(zip(class_names, sns.color_palette("tab10", n_classes)))
palette["Normal"] = (0.8, 0.8, 0.8)
flow_class = class_names[train_df["label"].to_numpy()]
ip_order = lambda ip: tuple(int(p) for p in ip.split("."))
short_ip = lambda ip: ".".join(ip.split(".")[-2:])


# View 1 - degree distribution: a few giant hubs, a long tail of tiny nodes.
deg = (G_train["ptr"][1:] - G_train["ptr"][:-1]).numpy()
plt.figure(figsize=(8, 4.5))
plt.hist(np.log10(deg), bins=60, color="steelblue")
plt.yscale("log")
plt.xlabel("log10(node degree)   (1 = node with one flow, 3 = node with 1000 flows)")
plt.ylabel("number of nodes (log scale)")
plt.title("Node degree distribution: a few giant hubs, a long tail of tiny nodes")
plt.tight_layout()
plt.show()

**View 2 - host-level graph.** Collapse every port: one node per IP, one arrow per source -> destination pair that carried attacks. Arrow width = number of attack flows, arrow colour = the most common attack type on that pair, node colour = subnet. This shows *who attacks whom*; because each arrow shows only its most common attack, rare classes do not appear here (see view 2b).

In [ ]:
flows = pd.DataFrame({"src": train_graph["srcip"].to_numpy(), "dst": train_graph["dstip"].to_numpy(), "cls": flow_class})
attacks = flows[flows["cls"] != "Normal"]
pairs = attacks.groupby(["src", "dst"])["cls"].agg(count="size", top=lambda s: s.value_counts().idxmax())
pairs = pairs.sort_values("count", ascending=False).head(120)

H = nx.DiGraph()
for (s, d), row in pairs.iterrows():
    H.add_edge(s, d, count=row["count"], top=row["top"])

subnet = lambda ip: ip.rsplit(".", 1)[0]
subnets = sorted({subnet(n) for n in H.nodes})
subnet_color = dict(zip(subnets, sns.color_palette("Set2", len(subnets))))
pos = nx.spring_layout(H, seed=SEED, k=1.2)

fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_nodes(H, pos, node_color=[subnet_color[subnet(n)] for n in H.nodes], node_size=450, edgecolors="black", ax=ax)
nx.draw_networkx_labels(H, pos, labels={n: n.split(".")[-1] for n in H.nodes}, font_size=8, ax=ax)
nx.draw_networkx_edges(H, pos, width=[0.5 + np.log10(d["count"] + 1) for _, _, d in H.edges(data=True)],
                       edge_color=[palette[d["top"]] for _, _, d in H.edges(data=True)],
                       arrows=True, arrowsize=10, alpha=0.8, connectionstyle="arc3,rad=0.1", ax=ax)
class_handles = [plt.Line2D([], [], color=palette[c], lw=3, label=c) for c in class_names if c != "Normal"]
subnet_handles = [plt.Line2D([], [], marker="o", ls="", color=c, markeredgecolor="black", label=f"{s}.x") for s, c in subnet_color.items()]
ax.legend(handles=class_handles + subnet_handles, loc="upper left", bbox_to_anchor=(1, 1), title="attack class / subnet")
ax.set_title(f"Host-level attack graph: {len(pairs)} attacker -> target pairs "
             f"(arrow colour = most common attack, label = last IP octet)")
ax.axis("off")
plt.tight_layout()
plt.show()

**View 2b - who sends which attack (rare classes included).** One panel per attack class: rows = source IPs, columns = target IPs, colour = number of flows (log scale, darker = more). Empty panels cells mean that pair never carried that attack.

In [ ]:
attack_classes = [c for c in class_names if c != "Normal"]
src_ips = sorted(attacks["src"].unique(), key=ip_order)
dst_ips = sorted(attacks["dst"].unique(), key=ip_order)

fig, axes = plt.subplots(3, 3, figsize=(18, 9))
for ax, c in zip(axes.flat, attack_classes):
    m = (attacks[attacks["cls"] == c].groupby(["src", "dst"]).size().unstack(fill_value=0)
         .reindex(index=src_ips, columns=dst_ips, fill_value=0))
    sns.heatmap(np.log10(m + 1), ax=ax, cmap="Reds", cbar=False, linewidths=0.2, linecolor="white",
                xticklabels=[short_ip(ip) for ip in dst_ips], yticklabels=[short_ip(ip) for ip in src_ips])
    ax.set_title(f"{c}  ({int(m.values.sum())} flows)", fontsize=11)
    ax.set_xlabel("target IP")
    ax.set_ylabel("source IP")
    ax.tick_params(labelsize=6)
plt.suptitle("Which source IP sends which attack to which target IP (train graph)", fontsize=13)
plt.tight_layout()
plt.show()

**View 3 - hubs at the endpoint level.** Back to (IP, port) nodes. One panel per endpoint among the three that touch the most attack flows: the endpoint in the middle (gold), one black dot per sampled flow around it (up to 35 attack + 15 other flows). The title gives the endpoint, its total flows and its **class mix over all its flows** - this is the neighbourhood the GNN averages over. A mixed title means averaging blurs the classes; a one-class title means neighbours agree.

In [ ]:
y_tr = G_train["y"]
all_nodes = torch.cat([G_train["src"], G_train["dst"]])
is_attack = torch.cat([(y_tr != normal_label)] * 2).float()
attack_per_node = torch.bincount(all_nodes, weights=is_attack, minlength=G_train["n_nodes"])
hubs = attack_per_node.argsort(descending=True)[:3].tolist()

rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5))
for ax, hub in zip(axes, hubs):
    lo, hi = G_train["ptr"][hub].item(), G_train["ptr"][hub + 1].item()
    edges = G_train["inc_edge"][lo:hi].numpy()
    is_att = y_tr[edges].numpy() != normal_label
    picked = np.concatenate([rng.choice(edges[is_att], min(is_att.sum(), 35), replace=False),
                             rng.choice(edges[~is_att], min((~is_att).sum(), 15), replace=False)])
    S = nx.Graph()
    for e in picked:
        S.add_edge(("hub", hub), ("flow", int(e)), cls=class_names[y_tr[e]])
    leaves = [n for n in S.nodes if n[0] == "flow"]
    pos = nx.shell_layout(S, nlist=[[("hub", hub)], leaves])
    nx.draw_networkx_edges(S, pos, edge_color=[palette[d["cls"]] for _, _, d in S.edges(data=True)], width=2, ax=ax)
    nx.draw_networkx_nodes(S, pos, nodelist=leaves, node_size=25, node_color="black", ax=ax)
    nx.draw_networkx_nodes(S, pos, nodelist=[("hub", hub)], node_size=600, node_color="gold", edgecolors="black", ax=ax)
    mix = pd.Series(class_names[y_tr[edges].numpy()]).value_counts(normalize=True).head(4)
    ax.set_title(f"{G_train['node_name'](hub)}\n{len(edges)} flows, {int(attack_per_node[hub])} attacks\n"
                 + ", ".join(f"{c} {p:.0%}" for c, p in mix.items()), fontsize=10)
    ax.axis("off")
fig.legend(handles=[plt.Line2D([], [], color=palette[c], lw=3, label=c) for c in class_names],
           loc="lower center", ncol=n_classes, frameon=False, title="flow class (edge colour)")
plt.suptitle("Neighbourhood of the 3 busiest attack endpoints", fontsize=13)
plt.tight_layout(rect=[0, 0.08, 1, 0.97])
plt.show()

**View 4 - do neighbouring flows share a class?** For a random sample of flows, pick one *other* flow that touches the same endpoint and check if it has the same class. If, for a class, that rate is far above the class's overall share, neighbours carry information about it - the reason a GNN can help. A class with no orange bar had **no** same-class neighbour in the sample. The table below the chart also shows how many sampled flows had a neighbour at all.

In [ ]:
sample = torch.randperm(len(y_tr))[:300_000]
hits, tries = np.zeros(n_classes), np.zeros(n_classes)
for side in ["src", "dst"]:
    nb, _ = sample_neighbors(G_train, G_train[side][sample], 1)
    nb = nb[:, 0]
    ok = nb != sample                                    # skip draws that picked the flow itself
    own, other = y_tr[sample][ok].numpy(), y_tr[nb][ok].numpy()
    tries += np.bincount(own, minlength=n_classes)
    hits += np.bincount(own[own == other], minlength=n_classes)

agreement = pd.DataFrame({"same-class neighbour rate": hits / np.maximum(tries, 1),
                          "class share of all flows": np.bincount(y_tr.numpy(), minlength=n_classes) / len(y_tr)},
                         index=class_names)
ax = agreement.plot.bar(figsize=(10, 4.5), logy=True, color=["darkorange", "lightgray"])
ax.set_ylim(5e-5, 2)
for i, rate in enumerate(agreement["same-class neighbour rate"]):
    if rate == 0:
        ax.text(i - 0.1, 3e-4, "none", ha="center", fontsize=9, color="darkorange", fontweight="bold")
plt.ylabel("rate (log scale)")
plt.title("Do neighbouring flows share a class? (orange far above gray = neighbours are informative)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print(agreement.assign(**{"sampled flows with a neighbour": tries.astype(int)}).round(4).to_string())

## 4. Model

A 2-layer E-GraphSAGE. Nodes start with a constant feature, so everything a node knows comes from the flows around it:
1. **Layer 1:** a node's embedding = mean of a transform of its incident edges' features.
2. **Layer 2:** the same idea one hop further - a node aggregates its neighbours' layer-1 embeddings together with the connecting edges' features.
3. **Edge classifier:** `[source embedding, destination embedding, the edge's own features]` -> MLP -> 10 classes. The edge's own features are included so the GNN can never do worse than the tabular view of the same flow.

Neighbours are **sampled** (8 per node in layer 2, 5 in layer 1, with replacement) - the standard GraphSAGE trick that keeps hub nodes with huge degree affordable and allows mini-batch training. At evaluation, predictions are averaged over several sampling passes.

In [ ]:
class EdgeSAGE(nn.Module):
    def __init__(self, n_feat, hidden, n_classes):
        super().__init__()
        self.edge1 = nn.Linear(n_feat, hidden)
        self.upd1 = nn.Linear(1 + hidden, hidden)
        self.node2 = nn.Linear(hidden, hidden, bias=False)
        self.edge2 = nn.Linear(n_feat, hidden)
        self.upd2 = nn.Linear(2 * hidden, hidden)
        self.head = nn.Sequential(nn.Linear(2 * hidden + n_feat, 128), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(128, n_classes))

    def layer1(self, G, nodes):
        eids, _ = sample_neighbors(G, nodes, FANOUT_2)
        msg = torch.relu(self.edge1(G["x"][eids]))                       # (nodes, fanout, hidden)
        ones = torch.ones(len(nodes), 1, device=nodes.device)             # constant initial node feature
        return torch.relu(self.upd1(torch.cat([ones, msg.mean(1)], dim=1)))

    def forward(self, G, idx):
        ends = torch.cat([G["src"][idx], G["dst"][idx]])
        h1_self = self.layer1(G, ends)
        eids, nbrs = sample_neighbors(G, ends, FANOUT_1)
        h1_nbr = self.layer1(G, nbrs.reshape(-1)).view(len(ends), FANOUT_1, -1)
        msg = torch.relu(self.node2(h1_nbr) + self.edge2(G["x"][eids]))
        h2 = torch.relu(self.upd2(torch.cat([h1_self, msg.mean(1)], dim=1)))
        h_src, h_dst = h2[:len(idx)], h2[len(idx):]
        return self.head(torch.cat([h_src, h_dst, G["x"][idx]], dim=1))


G_train = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in G_train.items()}
G_test = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in G_test.items()}
model = EdgeSAGE(len(feature_cols), HIDDEN, n_classes).to(DEVICE)
print("parameters:", sum(p.numel() for p in model.parameters()))

## 5. Train

- 15% of the **train edges** are held out as a validation set (their labels are hidden from the loss; their features still sit in the graph). The test graph is not touched until the end.
- Class weights are the square root of the "balanced" weights. Fully balanced weights would give Worms (137 train rows) a ~1000x weight and destabilise training; the square root still lifts rare classes without that.
- Early stopping on validation **macro F1**, the metric used to judge the baselines.

In [ ]:
y_all = G_train["y"].cpu().numpy()
tr_np, va_np = train_test_split(np.arange(len(y_all)), test_size=0.15, stratify=y_all, random_state=SEED)
tr_idx, va_idx = torch.from_numpy(tr_np).to(DEVICE), torch.from_numpy(va_np).to(DEVICE)

cw = np.sqrt(compute_class_weight("balanced", classes=np.arange(n_classes), y=y_all[tr_np]))
class_weights = torch.tensor(cw, dtype=torch.float32, device=DEVICE)
print({n: round(float(w), 1) for n, w in zip(class_names, cw)})

opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)


@torch.no_grad()
def predict_proba(G, idx, passes=1, batch=8192):
    model.eval()
    out = []
    for i in range(0, len(idx), batch):
        b = idx[i:i + batch]
        out.append((sum(torch.softmax(model(G, b), 1) for _ in range(passes)) / passes).cpu())
    return torch.cat(out)


history, best, bad_epochs = [], {"f1": -1.0, "state": None, "epoch": 0}, 0
y_val = G_train["y"][va_idx].cpu()
for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    model.train()
    perm = tr_idx[torch.randperm(len(tr_idx), device=DEVICE)]
    losses = []
    for i in range(0, len(perm), BATCH):
        b = perm[i:i + BATCH]
        opt.zero_grad()
        loss = loss_fn(model(G_train, b), G_train["y"][b])
        loss.backward()
        opt.step()
        losses.append(loss.item())

    val_proba = predict_proba(G_train, va_idx)
    val_loss = F.nll_loss(torch.log(val_proba + 1e-9), y_val, weight=class_weights.cpu()).item()
    val_f1 = f1_score(y_val.numpy(), val_proba.argmax(1).numpy(), average="macro")
    history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_loss": val_loss, "val_macro_f1": val_f1})
    print(f"epoch {epoch:>2} | train loss {np.mean(losses):.4f} | val loss {val_loss:.4f} | "
          f"val macro F1 {val_f1:.4f} | {time.time() - t0:.0f}s")

    if val_f1 > best["f1"]:
        best, bad_epochs = {"f1": val_f1, "state": copy.deepcopy(model.state_dict()), "epoch": epoch}, 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("early stop")
            break

model.load_state_dict(best["state"])
print(f"best epoch: {best['epoch']} (val macro F1 {best['f1']:.4f})")

### Training curves

In [ ]:
hist = pd.DataFrame(history).set_index("epoch")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hist[["train_loss", "val_loss"]].plot(ax=axes[0], marker="o")
axes[0].set_ylabel("weighted cross-entropy")
axes[0].set_title("GNN - training / validation loss")
hist["val_macro_f1"].plot(ax=axes[1], marker="o", color="darkorange")
axes[1].axvline(best["epoch"], color="gray", ls="--", label="best epoch")
axes[1].set_ylabel("macro F1")
axes[1].legend()
axes[1].set_title("GNN - validation macro F1")
plt.tight_layout()
plt.show()

## 6. Evaluate on the held-out test graph

The test graph is built only from test flows. Same metrics and confusion-matrix format as the baseline notebook; predictions are averaged over 5 neighbour-sampling passes.

In [ ]:
test_idx = torch.arange(len(G_test["src"]), device=DEVICE)
gnn_proba = predict_proba(G_test, test_idx, passes=5).numpy()
y_test = G_test["y"].cpu().numpy()
labels = np.arange(n_classes)


def report(name, proba):
    pred = proba.argmax(1)
    print(f"{name} | accuracy: {accuracy_score(y_test, pred):.4f} | macro F1: {f1_score(y_test, pred, average='macro'):.4f}")
    print(classification_report(y_test, pred, labels=labels, target_names=class_names, digits=3, zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=labels, normalize="true")
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names,
                cbar_kws={"label": "share of true class"})
    plt.xlabel("predicted")
    plt.ylabel("true")
    plt.title(f"{name} - confusion matrix (row-normalized, test set)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


report("GNN", gnn_proba)

## 7. Fusion with the tabular models

The GNN and the tabular models make different mistakes, so averaging their class probabilities can beat either alone. The baseline notebook saved its test probabilities (indexed by `row_id`, so rows line up exactly).

**The fusion weight is fixed at 50/50 before looking at any result** - it is not tuned on the test set. The sensitivity curve below is only there to show how much the weight matters; it is not used to pick the headline number.

In [ ]:
tab_file = glob.glob("/kaggle/input/**/two_stage_test_proba.parquet", recursive=True)[0]
tab_proba = pd.read_parquet(tab_file).loc[test_df.index].to_numpy("float32")   # aligned by row_id

fused_proba = 0.5 * tab_proba + 0.5 * gnn_proba
models = {"Two-Stage (tabular)": tab_proba, "GNN": gnn_proba, "Fusion 50/50": fused_proba}
scores = {name: f1_score(y_test, p.argmax(1), average="macro") for name, p in models.items()}
print({k: round(v, 4) for k, v in scores.items()})

per_class = pd.DataFrame({name: f1_score(y_test, p.argmax(1), labels=labels, average=None, zero_division=0)
                          for name, p in models.items()}, index=class_names)
per_class.plot.bar(figsize=(11, 4), width=0.8)
plt.ylabel("F1")
plt.title("Per-class F1 on the test set: tabular vs GNN vs fusion")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

weights = np.linspace(0, 1, 11)
sens = [f1_score(y_test, ((1 - w) * tab_proba + w * gnn_proba).argmax(1), average="macro") for w in weights]
plt.figure(figsize=(7, 4))
plt.plot(weights, sens, marker="o")
plt.axvline(0.5, color="gray", ls="--", label="pre-chosen weight 0.5")
plt.xlabel("weight on GNN (0 = tabular only, 1 = GNN only)")
plt.ylabel("macro F1")
plt.title("Fusion weight sensitivity (for information only)")
plt.legend()
plt.tight_layout()
plt.show()

report("Fusion 50/50", fused_proba)

## 8. Final comparison

Baseline macro F1 values are from the baseline notebook run that produced the saved probabilities (same test set). Rare classes are tiny (Worms = 34 test rows), so single-run macro F1 moves by about +/-0.03 between otherwise identical runs - differences smaller than that are not meaningful.

In [ ]:
baseline_macro_f1 = {
    "Random Forest": 0.6232, "LightGBM": 0.5734, "XGBoost": 0.6393,
    "Neural Network": 0.5357, "Two-Stage (LightGBM)": 0.6540, "Ensemble": 0.6281,
}
compare = pd.Series({**baseline_macro_f1, "GNN (E-GraphSAGE)": scores["GNN"], "Fusion: Two-Stage + GNN": scores["Fusion 50/50"]}).sort_values()
colors = ["darkorange" if n.startswith(("GNN", "Fusion")) else "steelblue" for n in compare.index]
plt.figure(figsize=(8, 4.5))
plt.barh(compare.index, compare.values, color=colors)
for i, v in enumerate(compare.values):
    plt.text(v + 0.003, i, f"{v:.3f}", va="center")
plt.xlabel("macro F1 (test set)")
plt.title("All models on the same test set")
plt.xlim(0, max(compare.max() + 0.08, 0.75))
plt.tight_layout()
plt.show()

torch.save(best["state"], "/kaggle/working/gnn_model.pt")
np.save("/kaggle/working/gnn_test_proba.npy", gnn_proba)
print("saved gnn_model.pt and gnn_test_proba.npy")